### Preparación del Entorno

In [0]:
# Cargamos la tabla limpia y la convertimos en una Vista Temporal de Spark
df_citas = spark.table("default.citas_pmm_limpioV3")
df_citas.createOrReplaceTempView("v_citas_pmm")

print("Vista 'v_citas_pmm' creada exitosamente. Lista para análisis SQL.")

Vista 'v_citas_pmm' creada exitosamente. Lista para análisis SQL.


### N1: Rendimiento Financiero y Distribución de Cobertura por Aseguradora

In [0]:
%sql
CREATE OR REPLACE TABLE rendimiento_aseguradora AS
SELECT
    nom_sucursal AS Sucursal,
    especialidad_medica AS Especialidad,
    nom_aseguradora AS Aseguradora,
    COUNT(*) AS Volumen_Citas,
    ROUND(SUM(pago_aseg), 2) AS Total_Pagado_Aseguradora_USD,
    ROUND(SUM(pago_clie), 2) AS Total_Copago_Paciente_USD,
    ROUND(SUM(pago_total), 2) AS Ingresos_Totales_USD,
    ROUND((SUM(pago_aseg) / SUM(pago_total)) * 100, 2) AS Porcentaje_Cobertura_Promedio
FROM v_citas_pmm
WHERE estado_cita = 'Completada'
GROUP BY nom_aseguradora, especialidad_medica, nom_sucursal
ORDER BY Ingresos_Totales_USD DESC;

SELECT * FROM rendimiento_aseguradora;

Sucursal,Especialidad,Aseguradora,Volumen_Citas,Total_Pagado_Aseguradora_USD,Total_Copago_Paciente_USD,Ingresos_Totales_USD,Porcentaje_Cobertura_Promedio
PMM Costa del Este,Radiología,Particular / Sin Seguro,52,0.0,4893.1,4893.1,0.0
PMM San Francisco,Neurología,Particular / Sin Seguro,37,0.0,4788.03,4788.03,0.0
PMM San Francisco,Otorrinolaringología,Particular / Sin Seguro,61,0.0,4533.93,4533.93,0.0
PMM San Francisco,Radiología,Particular / Sin Seguro,45,0.0,4101.61,4101.61,0.0
PMM San Francisco,Gastroenterología,Particular / Sin Seguro,37,0.0,4022.82,4022.82,0.0
PMM Costa del Este,Oftalmología,Particular / Sin Seguro,47,0.0,3960.88,3960.88,0.0
PMM Costa del Este,Dermatología,Particular / Sin Seguro,45,0.0,3938.82,3938.82,0.0
PMM San Francisco,Psiquiatría,Particular / Sin Seguro,39,0.0,3759.29,3759.29,0.0
PMM Costa del Este,Fisioterapia,Particular / Sin Seguro,53,0.0,3734.74,3734.74,0.0
PMM San Francisco,Radiología,PALIG,40,2516.5,1054.14,3570.64,70.48


### N2: Tasa de Cancelación por Sucursal

In [0]:
%sql
CREATE OR REPLACE TABLE cancelacion_sucursal AS
SELECT 
    nom_sucursal AS Sucursal,
    especialidad_medica AS Especialidad,
    COUNT(*) AS Citas_Agendadas,
    SUM(CASE WHEN estado_cita = 'Cancelada' THEN 1 ELSE 0 END) AS Citas_Canceladas,
    SUM(CASE WHEN estado_cita = 'Completada' THEN 1 ELSE 0 END) AS Citas_Completadas,
    ROUND((SUM(CASE WHEN estado_cita = 'Cancelada' THEN 1 ELSE 0 END) / COUNT(*)) * 100, 2) AS Tasa_Cancelacion_PCT
FROM v_citas_pmm
GROUP BY nom_sucursal, especialidad_medica
ORDER BY Tasa_Cancelacion_PCT DESC;
SELECT * FROM cancelacion_sucursal;

Sucursal,Especialidad,Citas_Agendadas,Citas_Canceladas,Citas_Completadas,Tasa_Cancelacion_PCT
PMM San Francisco,Geriatría,35,12,23,34.29
PMM El Dorado,Pediatría,51,17,34,33.33
PMM Costa del Este,Cardiología,120,35,85,29.17
PMM Tocumen,Gastroenterología,63,17,46,26.98
PMM Costa del Este,Geriatría,42,11,31,26.19
PMM Tocumen,Nefrología,66,17,49,25.76
PMM Brisas del Golf,Cardiología,66,17,49,25.76
PMM Brisas del Golf,Oftalmología,140,35,105,25.0
PMM El Dorado,Ginecología,60,15,45,25.0
PMM Tocumen,Neurología,45,11,34,24.44


### N3: Segmentación Demográfica y Ticket Promedio

In [0]:
%sql
CREATE OR REPLACE TABLE segmentacion_demografica AS
SELECT 
    nom_sucursal AS Sucursal,
    especialidad_medica AS Especialidad,
    CASE 
        WHEN edad_pac_cita < 15 THEN 'Pediátrico (<15 años)'
        WHEN edad_pac_cita BETWEEN 15 AND 64 THEN 'Adulto (15-64 años)'
        ELSE 'Geriátrico (65+ años)' 
    END AS Segmento_Poblacional,
    COUNT(*) AS Total_Atenciones,
    ROUND(SUM(pago_total), 2) AS Facturacion_Total_USD,
    ROUND(AVG(pago_total), 2) AS Ticket_Promedio_USD
FROM v_citas_pmm
WHERE estado_cita = 'Completada'
GROUP BY 
    CASE 
        WHEN edad_pac_cita < 15 THEN 'Pediátrico (<15 años)'
        WHEN edad_pac_cita BETWEEN 15 AND 64 THEN 'Adulto (15-64 años)'
        ELSE 'Geriátrico (65+ años)' 
    END, 
    nom_sucursal,
    especialidad_medica
    ORDER BY Segmento_Poblacional ASC;
SELECT * FROM segmentacion_demografica;

Sucursal,Especialidad,Segmento_Poblacional,Total_Atenciones,Facturacion_Total_USD,Ticket_Promedio_USD
PMM El Dorado,Otorrinolaringología,Adulto (15-64 años),77,5159.64,67.01
PMM Costa del Este,Nefrología,Adulto (15-64 años),80,9232.89,115.41
PMM El Dorado,Dermatología,Adulto (15-64 años),78,5133.28,65.81
PMM San Francisco,Psiquiatría,Adulto (15-64 años),95,8702.94,91.61
PMM El Dorado,Neurología,Adulto (15-64 años),68,6795.7,99.94
PMM Costa del Este,Cardiología,Adulto (15-64 años),57,6301.65,110.56
PMM San Francisco,Ginecología,Adulto (15-64 años),64,4605.49,71.96
PMM Brisas del Golf,Ginecología,Adulto (15-64 años),26,1620.56,62.33
PMM Tocumen,Medicina General,Adulto (15-64 años),42,1225.3,29.17
PMM El Dorado,Ginecología,Adulto (15-64 años),37,2282.95,61.7


### N4: Rentabilidad de cada Especialidad por Hora

In [0]:
%sql
CREATE OR REPLACE TABLE rentabilidad_especialidad AS
SELECT 
    nom_sucursal AS Sucursal,
    especialidad_medica AS Especialidad,
    COUNT(*) AS Total_Consultas,
    ROUND(SUM(mins_cit) / 60, 2) AS Horas_Totales_Invertidas,
    ROUND(SUM(pago_total), 2) AS Ingresos_Totales_USD,
    ROUND(SUM(pago_total) / (SUM(mins_cit) / 60), 2) AS Rentabilidad_Por_Hora_USD
FROM v_citas_pmm
WHERE estado_cita = 'Completada'
GROUP BY especialidad_medica, nom_sucursal
ORDER BY Rentabilidad_Por_Hora_USD DESC;
SELECT * FROM rentabilidad_especialidad;

Sucursal,Especialidad,Total_Consultas,Horas_Totales_Invertidas,Ingresos_Totales_USD,Rentabilidad_Por_Hora_USD
PMM Costa del Este,Neurología,107,67.25,13886.95,206.5
PMM San Francisco,Neurología,144,88.75,17801.94,200.59
PMM Costa del Este,Nefrología,110,65.5,12851.98,196.21
PMM Costa del Este,Cardiología,85,51.25,9946.24,194.07
PMM San Francisco,Nefrología,123,72.0,13484.3,187.28
PMM San Francisco,Cardiología,102,63.0,11456.24,181.85
PMM Costa del Este,Gastroenterología,110,70.25,12354.81,175.87
PMM Brisas del Golf,Neurología,62,36.25,6133.79,169.21
PMM El Dorado,Neurología,102,64.5,10681.46,165.6
PMM Costa del Este,Psiquiatría,112,71.5,11625.93,162.6


### N5: Distribución de Tiempos y Edades

In [0]:
%sql
CREATE OR REPLACE TABLE histograma_consultas AS
SELECT 
    COUNT(*) AS Citas,
    edad_pac_cita AS Edad
FROM v_citas_pmm
WHERE estado_cita = 'Completada'
GROUP BY edad_pac_cita;
SELECT * FROM histograma_consultas;

Citas,Edad
40,93
44,67
82,38
47,70
123,12
105,18
71,74
93,64
50,94
82,16


### N6: Burbujas de Edad, IMC, Pagos y Sexo

In [0]:
%sql
CREATE OR REPLACE TABLE burbujas_salud_costo AS
SELECT 
    edad_pac_cita AS Edad,
    imc AS IMC,
    pago_total AS PagoTotal,
    sexo_pac AS Sexo,
    especialidad_medica AS Especialidad,
    nom_sucursal AS Sucursal
FROM v_citas_pmm
WHERE estado_cita = 'Completada';
SELECT * FROM burbujas_salud_costo;

Edad,IMC,PagoTotal,Sexo,Especialidad,Sucursal
19,29.8,101.72,F,Otorrinolaringología,PMM El Dorado
44,21.6,50.89,M,Otorrinolaringología,PMM El Dorado
13,19.1,58.47,F,Odontología,PMM Costa del Este
65,21.8,65.5,M,Psiquiatría,PMM Tocumen
93,21.4,31.07,F,Medicina General,PMM San Francisco
81,19.7,84.31,F,Gastroenterología,PMM San Francisco
7,15.6,83.65,F,Dermatología,PMM El Dorado
29,27.3,63.34,M,Radiología,PMM El Dorado
7,15.8,56.49,F,Radiología,PMM El Dorado
71,29.0,47.4,F,Fisioterapia,PMM Brisas del Golf


### N7: Evolución de Aseguradoras

In [0]:
%sql
CREATE OR REPLACE TABLE area_aseguradoras AS
SELECT 
    date_format(CAST(fecha_cit AS Date), 'yyyy-MM') AS mes_cita,
    nom_aseguradora,
    ROUND(SUM(pago_total), 2) AS ingresos_totales
FROM v_citas_pmm
WHERE estado_cita = 'Completada'
GROUP BY 1, 2
ORDER BY mes_cita ASC;
SELECT * FROM area_aseguradoras;

mes_cita,nom_aseguradora,ingresos_totales
2025-01,Particular / Sin Seguro,11514.87
2025-01,Compañía Internacional de Seguros (IS),8025.38
2025-01,Seguros ASSA,9207.46
2025-01,PALIG,6769.03
2025-01,MAPFRE Panamá,6717.94
2025-01,Seguros SURA,7688.72
2025-02,PALIG,8238.61
2025-02,Compañía Internacional de Seguros (IS),7294.7
2025-02,Seguros SURA,6998.05
2025-02,MAPFRE Panamá,7482.89


### N8: Matriz de Horas Pico

In [0]:
%sql
CREATE OR REPLACE TABLE heatmap_horas_pico AS
SELECT 
    date_format(CAST(fecha_cit AS DATE), 'EEEE') AS Dia_semana,
    CAST(dayofweek(CAST(fecha_cit AS DATE)) AS INT) AS Num_dia_semana,
    date_format(CAST(hr_inicio_cit AS TIMESTAMP), 'HH') AS Hora_inicio,
    COUNT(*) AS total_citas
FROM v_citas_pmm
GROUP BY 1, 2, 3
ORDER BY Num_dia_semana ASC, Hora_inicio ASC;
SELECT * FROM heatmap_horas_pico;

Dia_semana,Num_dia_semana,Hora_inicio,total_citas
Sunday,1,08,38
Sunday,1,09,29
Sunday,1,10,28
Sunday,1,11,7
Sunday,1,12,12
Sunday,1,13,7
Sunday,1,14,16
Sunday,1,15,26
Sunday,1,16,30
Sunday,1,17,17
